# Violence Detection with MoViNet-A2 (Colab, Simplified)

This notebook fine-tunes **MoViNet-A2** on the [Merged Violence Dataset](https://www.kaggle.com/datasets/messaoudiriham/merged-violence-dataset).

**Expected input layout (already prepared by you):**
```
/content/drive/MyDrive/Violence Detection Dataset/
    Violence/        <- video files
    NonViolence/      <- video files
```

There is **no train/val split yet** — this notebook creates one automatically (80/20) by copying files into a working folder, then trains MoViNet-A2 with data augmentation, same as the reference notebook but simplified.

**Steps:**
1. Install dependencies
2. Mount Google Drive
3. Auto-create train/val split
4. Build a `tf.data` pipeline with augmentation
5. Load MoViNet-A2 pretrained backbone, fine-tune
6. Train, evaluate, plot results, save model


## 1. Install dependencies

In [ ]:
# !pip install -q tf-models-official tensorflow_hub tensorflow-docs tensorflow remotezip tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.7/182.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.3 MB/s eta 0:00:00


## 2. Mount Google Drive

In [ ]:

!pip install tensorflow==2.19.0
!pip install tf-models-official==2.19.1

Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: tf-models-official 2.20.0
Uninstalling tf-models-official-2.20.0:
  Successfully uninstalled tf-models-official-2.20.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 784.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 134.0 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.20.0
    Uninstalling tensorboard-2.20.0:
      Successfully uninstalled tensorboard-2.20.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf-tf 2.20.0 requires tensorflow

INFO: pip is looking at multiple versions of tf-keras to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 28.1 MB/s eta 0:00:00
  Attempting uninstall: tf-keras
    Found existing installation: tf_keras 2.20.0
    Uninstalling tf_keras-2.20.0:
      Successfully uninstalled tf_keras-2.20.0
  Attempting uninstall: tensorflow-text
    Found existing installation: tensorflow-text 2.20.1
    Uninstalling tensorflow-text-2.20.1:
ERROR: Operation cancelled by user
^C


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 3. Set paths

Change `SOURCE_DATASET_DIR` if your folder lives somewhere else in Drive.


In [ ]:
import pathlib

# Folder that directly contains "Violence" and "NonViolence" subfolders
SOURCE_DATASET_DIR = pathlib.Path("/content/drive/MyDrive/Violence Detection Dataset")

# Working copy with train/val split (created on the Colab local disk for fast I/O)
OUTPUT_DIR = pathlib.Path("/content/output_dataset")

VIDEO_EXTS = (".avi", ".mp4", ".mov", ".mkv")

print("Violence videos found:   ", len([f for f in (SOURCE_DATASET_DIR/'Violence').glob('*') if f.suffix.lower() in VIDEO_EXTS]))
print("NonViolence videos found:", len([f for f in (SOURCE_DATASET_DIR/'NonViolence').glob('*') if f.suffix.lower() in VIDEO_EXTS]))


Violence videos found:    3500
NonViolence videos found: 3501


## 4. Create train / val split (80 / 20)

Only needs to run once. Uses `shutil.copy2` so your original Drive files are untouched.

In [ ]:
import os
import shutil
import random
from concurrent.futures import ThreadPoolExecutor

def create_dataset_splits(source_dir, output_dir, split_ratio=0.8, seed=42):
    categories = ['Violence', 'NonViolence']
    splits = ['train', 'val']

    for split in splits:

        for category in categories:
            os.makedirs(output_dir / split / category, exist_ok=True)

    random.seed(seed)
    tasks = []

    for category in categories:
        category_path = source_dir / category
        if not category_path.exists():
            print(f"Warning: '{category_path}' does not exist. Skipping.")
            continue

        files = [f for f in category_path.iterdir() if f.is_file() and f.suffix.lower() in VIDEO_EXTS]
        if not files:
            print(f"Warning: no video files found in '{category_path}'.")
            continue

        random.shuffle(files)
        split_index = int(len(files) * split_ratio)
        train_files, val_files = files[:split_index], files[split_index:]

        for f in train_files:
            tasks.append((f, output_dir / 'train' / category / f.name))
        for f in val_files:
            tasks.append((f, output_dir / 'val' / category / f.name))

    def _copy(pair):
        src, dst = pair
        if not dst.exists():
            shutil.copy2(src, dst)

    with ThreadPoolExecutor(max_workers=8) as ex:
        list(ex.map(_copy, tasks))

    print(f"Copied {len(tasks)} files into {output_dir}")

if not OUTPUT_DIR.exists():
    create_dataset_splits(SOURCE_DATASET_DIR, OUTPUT_DIR, split_ratio=0.8, seed=42)
else:
    print(f"{OUTPUT_DIR} already exists, skipping split (delete it to redo).")

subset_paths = {
    "train": OUTPUT_DIR / "train",
    "val":   OUTPUT_DIR / "val",
}

train_count = sum(len(list((subset_paths['train']/c).glob('*'))) for c in ['Violence', 'NonViolence'])
val_count   = sum(len(list((subset_paths['val']/c).glob('*'))) for c in ['Violence', 'NonViolence'])
print("Train videos:", train_count, "| Val videos:", val_count)


/content/output_dataset already exists, skipping split (delete it to redo).
Train videos: 5600 | Val videos: 1401


## 5. Imports

In [ ]:
!pip install -U tf-models-official==2.19.1

  Using cached tf_models_official-2.19.1-py2.py3-none-any.whl.metadata (1.6 kB)
  Using cached tensorflow_text-2.19.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.0 kB)
Using cached tf_models_official-2.19.1-py2.py3-none-any.whl (2.9 MB)
Using cached tensorflow_text-2.19.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (5.2 MB)


In [ ]:
import random
import datetime
import numpy as np
import pandas as pd
import cv2
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub

from tensorflow.keras.losses import SparseCategoricalCrossentropy
from official.projects.movinet.modeling import movinet
from official.projects.movinet.modeling import movinet_model

print("GPU:", tf.config.list_physical_devices('GPU'))


ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.12/dist-packages/tensorflow_datasets/co

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import tf_keras

EarlyStopping = tf_keras.callbacks.EarlyStopping
ModelCheckpoint = tf_keras.callbacks.ModelCheckpoint

## 6. Frame extraction + augmentation

`frames_from_video_file` samples `n_frames` uniformly spaced frames from each video (same approach as the reference notebook).
`augment_frames` applies random flip / brightness / contrast / saturation jitter — only used for the **training** split.


In [ ]:
def format_frames(frame, output_size):
    """Pad and resize a single frame to output_size (H, W)."""
    frame = tf.image.convert_image_dtype(frame, tf.float32)
    frame = tf.image.resize_with_pad(frame, *output_size)
    return frame


def frames_from_video_file(video_path, n_frames, output_size=(224, 224), frame_step=15):
    """Read n_frames spaced frame_step apart from a video, starting at a random offset."""
    result = []
    src = cv2.VideoCapture(str(video_path))

    video_length = int(src.get(cv2.CAP_PROP_FRAME_COUNT))   # <-- cast to int
    need_length = 1 + (n_frames - 1) * frame_step

    if need_length > video_length:
        start = 0
    else:
        max_start = video_length - need_length
        start = random.randint(0, max_start + 1)

    src.set(cv2.CAP_PROP_POS_FRAMES, start)
    ret, frame = src.read()
    result.append(format_frames(frame, output_size))

    for _ in range(n_frames - 1):
        for _ in range(frame_step):
            ret, frame = src.read()
        if ret:
            frame = format_frames(frame, output_size)
            result.append(frame)
        else:
            result.append(np.zeros_like(result[0]))
    src.release()
    result = np.array(result)[..., [2, 1, 0]]  # BGR -> RGB

    return result


def augment_frames(frames):
    """Apply the SAME random flip to every frame in the clip, and per-clip
    brightness/contrast/saturation jitter, so the augmentation stays temporally
    consistent across the whole video clip."""
    frames = tf.convert_to_tensor(frames)

    if tf.random.uniform([]) > 0.5:
        frames = tf.image.flip_left_right(frames)

    frames = tf.image.random_brightness(frames, max_delta=0.15)
    frames = tf.image.random_contrast(frames, lower=0.85, upper=1.15)
    frames = tf.image.random_saturation(frames, lower=0.85, upper=1.15)
    frames = tf.clip_by_value(frames, 0.0, 1.0)

    return frames.numpy()


## 7. Frame generator (yields `(frames, label)` pairs for `tf.data`)

In [ ]:
class FrameGenerator:
    def __init__(self, path, n_frames, training=False):
        self.path = path
        self.n_frames = n_frames
        self.training = training
        self.class_names = sorted(set(p.name for p in self.path.iterdir() if p.is_dir()))
        self.class_ids_for_name = dict((name, idx) for idx, name in enumerate(self.class_names))

    def get_files_and_class_names(self):
        video_paths = [p for p in self.path.glob('*/*') if p.suffix.lower() in VIDEO_EXTS]
        classes = [p.parent.name for p in video_paths]
        return video_paths, classes

    def __call__(self):
        video_paths, classes = self.get_files_and_class_names()
        pairs = list(zip(video_paths, classes))

        if self.training:
            random.shuffle(pairs)

        for path, name in pairs:
            video_frames = frames_from_video_file(path, self.n_frames)
            if self.training:
                video_frames = augment_frames(video_frames)
            label = self.class_ids_for_name[name]
            yield video_frames, label


fg = FrameGenerator(subset_paths['train'], 10, training=True)
frames, label = next(fg())
print("Sample clip shape:", frames.shape, "| label:", label)
print("Classes:", fg.class_ids_for_name)


Sample clip shape: (10, 224, 224, 3) | label: 0
Classes: {'NonViolence': 0, 'Violence': 1}


## 8. Build the `tf.data` datasets

In [ ]:
batch_size = 8
num_frames = 32   # lower than 64 to keep Colab GPU memory usage manageable; raise if you have headroom

output_signature = (
    tf.TensorSpec(shape=(None, None, None, 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int16),
)

train_ds = tf.data.Dataset.from_generator(
    FrameGenerator(subset_paths['train'], num_frames, training=True),
    output_signature=output_signature
)
val_ds = tf.data.Dataset.from_generator(
    FrameGenerator(subset_paths['val'], num_frames, training=False),
    output_signature=output_signature
)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.batch(batch_size).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.batch(batch_size).prefetch(buffer_size=AUTOTUNE)

for f, l in train_ds.take(1):
    print("Batch frames shape:", f.shape, "| batch labels:", l.numpy())


Batch frames shape: (8, 32, 224, 224, 3) | batch labels: [1 0 1 1 1 1 0 1]


## 9. Build MoViNet-A2 and load pretrained (Kinetics) weights

Downloads the official MoViNet-A2 base checkpoint, restores it into a 600-class
classifier head (to match the checkpoint), then rebuilds the classifier with
**2 classes** (Violence / NonViolence) for fine-tuning. The last 20% of backbone
layers are left trainable, the rest stay frozen.


In [ ]:
model_id = 'a2'
resolution = 224
num_classes = 2

tf.keras.backend.clear_session()

backbone = movinet.Movinet(model_id=model_id)
backbone.trainable = True

# Freeze all layers, then unfreeze last 20% for fine-tuning
n_layers = len(backbone.layers)
for layer in backbone.layers:
    layer.trainable = False
for layer in backbone.layers[int(n_layers * 0.8):]:
    layer.trainable = True

# Build a 600-class model to match the Kinetics pretrained checkpoint shape
model_for_ckpt = movinet_model.MovinetClassifier(backbone=backbone, num_classes=600)
model_for_ckpt.build([None, None, None, None, 3])

!wget -q https://storage.googleapis.com/tf_model_garden/vision/movinet/movinet_a2_base.tar.gz -O movinet_a2_base.tar.gz
!tar -xf movinet_a2_base.tar.gz

checkpoint_dir = f'movinet_{model_id}_base'
checkpoint_path = tf.train.latest_checkpoint(checkpoint_dir)
checkpoint = tf.train.Checkpoint(model=model_for_ckpt)
status = checkpoint.restore(checkpoint_path)
status.assert_existing_objects_matched()

print("Pretrained weights restored.")


Pretrained weights restored.


In [ ]:
def build_classifier(batch_size, num_frames, resolution, backbone, num_classes):
    """Builds the final classifier (2 classes) on top of the fine-tuned backbone."""
    model = movinet_model.MovinetClassifier(backbone=backbone, num_classes=num_classes)
    model.build([batch_size, num_frames, resolution, resolution, 3])
    return model


model_a2 = build_classifier(batch_size, num_frames, resolution, backbone, num_classes)
model_a2.summary()


Model: "movinet_classifier_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 image (InputLayer)          [(None, None, None, Non   0         
                             e, 3)]                              
                                                                 
 movinet (Movinet)           ({'stem': (None, None,    2738754   
                             None, None, 16),                    
                              'block0_layer0': (None             
                             , None, None, None, 16)             
                             , 'block0_layer1': (Non             
                             e, None, None, None, 16             
                             ),                                  
                              'block0_layer2': (None             
                             , None, None, None, 16)             
                             , 'block1_layer0'

## 10. Compile

In [ ]:
print(type(model_a2))

<class 'official.projects.movinet.modeling.movinet_model.MovinetClassifier'>


In [ ]:
import tensorflow as tf
import keras

print(tf.keras)
print(keras)

<module 'tensorflow.keras' from '/usr/local/lib/python3.12/dist-packages/keras/_tf_keras/keras/__init__.py'>
<module 'keras' from '/usr/local/lib/python3.12/dist-packages/keras/__init__.py'>


In [ ]:
num_epochs = 20

import tf_keras

optimizer = tf_keras.optimizers.Adam(
    learning_rate=1e-5
)

model_a2.compile(
    optimizer=optimizer,
    loss=tf_keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

## 11. Callbacks

In [ ]:
checkpoint_filepath = '/content/drive/MyDrive/movinet_a2_violence_best.keras'

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ModelCheckpoint(checkpoint_filepath, monitor='val_accuracy', save_best_only=True, save_weights_only=False),
]


## 12. Train

In [ ]:
with tf.device('/gpu:0'):
    results_a2 = model_a2.fit(
        train_ds,
        validation_data=val_ds,
        epochs=num_epochs,
        validation_freq=1,
        callbacks=callbacks,
        verbose=1,
    )


Epoch 1/20
700/700 [==============================] - 2974s 4s/step - loss: 0.4843 - accuracy: 0.7750 - val_loss: 0.2609 - val_accuracy: 0.9136
Epoch 2/20
700/700 [==============================] - 2859s 4s/step - loss: 0.2625 - accuracy: 0.9007 - val_loss: 0.2020 - val_accuracy: 0.9251
Epoch 3/20
254/700 [=========>....................] - ETA: 25:00 - loss: 0.2087 - accuracy: 0.9183

KeyboardInterrupt: 

## 13. Evaluate

In [ ]:
with tf.device('/gpu:0'):
    model_a2.evaluate(val_ds, return_dict=True)


     42/Unknown - 155s 4s/step - loss: 0.1847 - accuracy: 0.9256

KeyboardInterrupt: 

## 14. Confusion matrix

In [ ]:
def get_actual_predicted_labels(dataset, model):
    actual = []
    for _, labels in dataset.unbatch():
        actual.append(labels)
    actual = tf.stack(actual, axis=0)

    logits = model.predict(dataset, verbose=1)
    predicted = tf.argmax(logits, axis=1)
    return actual, predicted


def plot_confusion_matrix(actual, predicted, labels, ds_type):
    cm = tf.math.confusion_matrix(actual, predicted)
    ax = sns.heatmap(cm, annot=True, fmt='g')
    sns.set(rc={'figure.figsize': (8, 8)})
    sns.set(font_scale=1.2)
    ax.set_title('Confusion matrix - ' + ds_type)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.xaxis.set_ticklabels(labels)
    ax.yaxis.set_ticklabels(labels)
    plt.show()
    return cm


fg_val = FrameGenerator(subset_paths['val'], num_frames, training=False)
label_names = list(fg_val.class_ids_for_name.keys())

with tf.device('/gpu:0'):
    actual, predicted = get_actual_predicted_labels(val_ds, model_a2)

plot_confusion_matrix(actual, predicted, label_names, 'val')


KeyboardInterrupt: 

## 15. Plot training history

In [ ]:
def plot_history(history):
    h = history.history
    epochs = range(1, len(h['loss']) + 1)

    plt.figure()
    plt.plot(epochs, h['loss'], label='train loss')
    plt.plot(epochs, h['val_loss'], label='val loss')
    plt.title('Loss')
    plt.xlabel('Epoch'); plt.legend(); plt.show()

    plt.figure()
    plt.plot(epochs, h['accuracy'], label='train acc')
    plt.plot(epochs, h['val_accuracy'], label='val acc')
    plt.title('Accuracy')
    plt.xlabel('Epoch'); plt.legend(); plt.show()

plot_history(results_a2)


## 16. Save final model to Drive

In [ ]:
save_path = f"/content/drive/MyDrive/movinet_a2_violence_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}.h5"
model_a2.save(save_path)
print("Saved to:", save_path)
